# 05 Route Investment Analysis

## Purpose

This notebook creates an early route productivity analysis by combining METRO scheduled trip counts from GTFS with a small sample of route-level ridership values from the October 2024 ridership report.

The goal is to move beyond simple ridership totals and begin asking a more useful planning question: **which routes generate strong rider demand relative to the amount of service provided?**

## Inputs

- `data/raw/metro_gtfs/merged/trips.txt`
- Manually entered October 2024 weekday ridership values for five key bus routes:
  - Route 82 Westheimer
  - Route 46 Gessner
  - Route 25 Richmond
  - Route 2 Bellaire
  - Route 4 Beechnut

## Outputs

This notebook is exploratory and does not create the final project dataset. It helps validate the productivity metric later used in the investment-priority analysis.


## 1. Import libraries and load GTFS trips

The `trips.txt` file describes scheduled transit trips. Each row represents a scheduled trip and includes fields such as route ID, trip ID, direction, and shape ID. Counting trips by route gives an approximate measure of how much scheduled service METRO provides on each route.


In [ ]:
import pandas as pd

trips = pd.read_csv(
    "../data/raw/metro_gtfs/merged/trips.txt"
)

print("Trips shape:", trips.shape)
trips.head()


## 2. Count scheduled trips by route

Ridership alone does not tell the full story. A route with high ridership may also receive a very high amount of service. Counting scheduled trips allows us to compare rider demand against service supply.


In [ ]:
route_trip_counts = (
    trips.groupby("route_id")
    .size()
    .reset_index(name="scheduled_trips")
)

route_trip_counts.sort_values(
    "scheduled_trips",
    ascending=False
).head(20)


## 3. Filter to key high-ridership corridors

This project initially focuses on five major METRO bus corridors that were identified as important from the ridership report and GTFS route exploration. These routes provide a manageable first sample before expanding to the full system.


In [ ]:
key_routes = [82, 46, 25, 2, 4]

route_trip_counts[
    route_trip_counts["route_id"].isin(key_routes)
]


## 4. Enter sample ridership values

The following ridership values come from the October 2024 METRO ridership report. They represent average weekday boardings for the five selected bus routes.

This manual step is temporary. Later notebooks move toward cleaner processed datasets that can be reused by the final recommendation notebook and future interactive website.


In [ ]:
ridership = pd.DataFrame({
    "route_id": [82, 46, 25, 2, 4],
    "route_name": ["Westheimer", "Gessner", "Richmond", "Bellaire", "Beechnut"],
    "avg_weekday_boardings": [14054, 7670, 7489, 7590, 8743]
})

ridership


## 5. Calculate route productivity

This section joins ridership with scheduled trip counts and calculates a simple productivity metric:

`boardings_per_scheduled_trip = average weekday boardings / scheduled trips`

This metric is useful because it compares rider demand to service supplied. Higher values suggest that a route is carrying more riders per scheduled trip and may warrant closer evaluation for additional service or capital investment.


In [ ]:
analysis = ridership.merge(
    route_trip_counts,
    on="route_id",
    how="left"
)

analysis["boardings_per_scheduled_trip"] = (
    analysis["avg_weekday_boardings"] / analysis["scheduled_trips"]
)

analysis.sort_values(
    "boardings_per_scheduled_trip",
    ascending=False
)


## Summary

This notebook introduced an early productivity metric for comparing ridership demand against scheduled service. In the five-route sample, Beechnut and Westheimer show especially strong boardings per scheduled trip, while Westheimer also has the highest overall weekday ridership.

This analysis supports the broader project goal of identifying corridors where additional transit investment may provide strong public benefit. Later notebooks expand this approach by saving processed datasets, adding route geography, mapping population density, and building a final investment-priority ranking.
